# CSE-CIC-IDS2018 Preprocessing Notebook

This notebook is used for Stage 1 data exploration and preprocessing experiments.

The final reusable preprocessing logic should be kept in:

`stage-1/scripts/sample_cse_cic_ids2018.py`

Large raw CSV files should not be committed to GitHub.

## Dataset source note

Primary Colab source:

`kagglehub.dataset_download("solarmainframe/ids-intrusion-csv")`

Local fallback source:

`archive.zip`

The local `archive.zip` was inspected before this notebook was updated. It contains 10 CSV files named by date, such as `02-14-2018.csv`, `02-20-2018.csv`, and `03-02-2018.csv`.

Most files contain 80 columns starting with `Dst Port`, `Protocol`, and `Timestamp`, ending with `Label`. The `02-20-2018.csv` file contains 84 columns and additionally includes `Flow ID`, `Src IP`, `Src Port`, and `Dst IP`. Because of this mixed schema, the notebook uses safe column access and fallback values when source or destination IP fields are missing.

## 1. Setup

In [ ]:
from pathlib import Path
import importlib.util
import json
import subprocess
import sys
import zipfile

import numpy as np
import pandas as pd

if importlib.util.find_spec('kagglehub') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'kagglehub'])

import kagglehub

KAGGLE_DATASET = 'solarmainframe/ids-intrusion-csv'
DATA_SOURCE = 'kagglehub'  # Use 'archive' to read a local archive.zip fallback.
ARCHIVE_PATH = Path('/content/archive.zip')
OUTPUT_PATH = Path('/content/sample-alerts.json')
# After reviewing the Colab output, copy the exported JSON into:
# stage-1/data/processed/sample-alerts.json
# Do not automatically overwrite the repository file from this notebook.
DATASET_DIR = None

BENIGN_SAMPLE_SIZE = 500
ATTACK_SAMPLE_SIZE = 500
TARGET_ATTACK_TYPES = [
    'Brute Force',
    'Heartbleed',
    'Botnet',
    'DoS',
    'DDoS',
    'Web Attack',
    'Infiltration',
]
CHUNK_SIZE = 200_000
RANDOM_STATE = 42

## 2. Load Dataset

In [ ]:
# Download through KaggleHub in Colab, or inspect archive.zip as a local fallback.
if DATA_SOURCE == 'kagglehub':
    dataset_path = kagglehub.dataset_download(KAGGLE_DATASET)
    DATASET_DIR = Path(dataset_path)
    csv_files = sorted(str(path.relative_to(DATASET_DIR)) for path in DATASET_DIR.rglob('*.csv'))
else:
    with zipfile.ZipFile(ARCHIVE_PATH) as archive:
        csv_files = [name for name in archive.namelist() if name.lower().endswith('.csv')]

print('Dataset path:', DATASET_DIR if DATA_SOURCE == 'kagglehub' else ARCHIVE_PATH)
csv_files

In [ ]:
# Use all available CSV files so the sample can include all available attack categories.
# For quick debugging, temporarily replace this with a smaller list.
SELECTED_FILES = csv_files

def read_csv_member(member_name, nrows=None, usecols=None, chunksize=None):
    if DATA_SOURCE == 'kagglehub':
        return pd.read_csv(DATASET_DIR / member_name, nrows=nrows, usecols=usecols, chunksize=chunksize)

    if chunksize is not None:
        raise ValueError('Use iter_csv_member for chunked archive reads.')

    with zipfile.ZipFile(ARCHIVE_PATH) as archive:
        with archive.open(member_name) as file:
            return pd.read_csv(file, nrows=nrows, usecols=usecols)

def iter_csv_member(member_name, usecols=None, chunksize=CHUNK_SIZE):
    if DATA_SOURCE == 'kagglehub':
        yield from pd.read_csv(DATASET_DIR / member_name, usecols=usecols, chunksize=chunksize)
        return

    with zipfile.ZipFile(ARCHIVE_PATH) as archive:
        with archive.open(member_name) as file:
            yield from pd.read_csv(file, usecols=usecols, chunksize=chunksize)

CSV_FILES_USED = SELECTED_FILES
CSV_FILES_USED

## 3. Inspect Columns

In [ ]:
# Preview one file before full sampling.
preview_df = read_csv_member(SELECTED_FILES[0], nrows=5)
print('Preview file:', SELECTED_FILES[0])
print('Preview rows, columns:', preview_df.shape)
print(preview_df.columns.tolist())
preview_df.head()

In [ ]:
# Inspect column differences across CSV files.
schema_summary = []
for name in csv_files:
    preview = read_csv_member(name, nrows=0)
    columns = preview.columns.tolist()
    schema_summary.append({
        'file': name,
        'column_count': len(columns),
        'has_src_ip': 'Src IP' in columns,
        'has_dst_ip': 'Dst IP' in columns,
        'has_label': 'Label' in columns,
        'first_columns': columns[:8],
    })

pd.DataFrame(schema_summary)

## 4. Clean Data

In [ ]:
def clean_column_names(dataframe):
    cleaned = dataframe.copy()
    cleaned.columns = cleaned.columns.str.strip()
    return cleaned

def clean_invalid_values(dataframe):
    cleaned = dataframe.copy()
    cleaned = cleaned.replace([np.inf, -np.inf], np.nan)
    if 'Label' in cleaned.columns:
        cleaned = cleaned[cleaned['Label'].astype(str).str.strip().str.lower() != 'label']
    if 'Dst Port' in cleaned.columns:
        cleaned = cleaned[cleaned['Dst Port'].astype(str).str.strip().str.lower() != 'dst port']
    cleaned = cleaned.dropna(subset=['Label'])
    return cleaned

# Cleaning is applied per chunk during sampling to avoid loading the full dataset into memory.
preview_df = clean_column_names(preview_df)
preview_df = clean_invalid_values(preview_df)
preview_df.head()

## 5. Check Label Distribution

In [ ]:
def audit_label_distribution(file_names):
    label_counts = {}
    rows_loaded = 0
    rows_after_cleaning = 0

    for file_name in file_names:
        for chunk in iter_csv_member(file_name, usecols=['Label'], chunksize=CHUNK_SIZE):
            rows_loaded += len(chunk)
            chunk = clean_column_names(chunk)
            chunk = clean_invalid_values(chunk)
            rows_after_cleaning += len(chunk)
            counts = chunk['Label'].astype(str).value_counts(dropna=False)
            for label, count in counts.items():
                label_counts[label] = label_counts.get(label, 0) + int(count)

    return label_counts, rows_loaded, rows_after_cleaning

LABEL_DISTRIBUTION_BEFORE_SAMPLING, ROWS_LOADED, ROWS_AFTER_CLEANING = audit_label_distribution(SELECTED_FILES)
ROWS_REMOVED_DURING_CLEANING = ROWS_LOADED - ROWS_AFTER_CLEANING
LABEL_DISTRIBUTION_BEFORE_SAMPLING

## 6. Sample Benign and Attack Rows

In [ ]:
def map_label_to_attack_type(label: str) -> str:
    normalized = str(label).strip().lower()
    compact = ' '.join(normalized.replace('-', ' ').replace('_', ' ').split())
    squashed = compact.replace(' ', '')
    if normalized == 'benign':
        return 'Benign'
    if 'heartbleed' in compact:
        return 'Heartbleed'
    if 'ddos' in compact:
        return 'DDoS'
    if 'dos' in compact:
        return 'DoS'
    if 'bot' in compact:
        return 'Botnet'
    if 'web attack' in compact or 'sql injection' in compact or 'xss' in compact:
        return 'Web Attack'
    if 'brute' in compact and 'web' in compact:
        return 'Web Attack'
    if 'ftp' in compact and ('brute' in compact or 'bruteforce' in squashed):
        return 'Brute Force'
    if 'ssh' in compact and ('brute' in compact or 'bruteforce' in squashed or 'patator' in compact):
        return 'Brute Force'
    if 'infilteration' in compact or 'infiltration' in compact:
        return 'Infiltration'
    return 'Unknown Attack'

def audit_attack_type_distribution(label_distribution):
    attack_type_counts = {}
    for label, count in label_distribution.items():
        attack_type = map_label_to_attack_type(label)
        attack_type_counts[attack_type] = attack_type_counts.get(attack_type, 0) + int(count)
    return attack_type_counts

ATTACK_TYPE_DISTRIBUTION_BEFORE_SAMPLING = audit_attack_type_distribution(LABEL_DISTRIBUTION_BEFORE_SAMPLING)

def sample_rows_by_attack_type(file_names, benign_n=BENIGN_SAMPLE_SIZE, attack_n=ATTACK_SAMPLE_SIZE):
    """Sample benign rows and attack rows stratified by mapped dashboard attackType."""
    available_attack_types = [
        attack_type for attack_type in TARGET_ATTACK_TYPES
        if ATTACK_TYPE_DISTRIBUTION_BEFORE_SAMPLING.get(attack_type, 0) > 0
    ]
    target_counts = {'Benign': min(benign_n, ATTACK_TYPE_DISTRIBUTION_BEFORE_SAMPLING.get('Benign', 0))}
    if available_attack_types:
        base_target = attack_n // len(available_attack_types)
        remainder = attack_n % len(available_attack_types)
        for index, attack_type in enumerate(available_attack_types):
            requested = base_target + (1 if index < remainder else 0)
            available = ATTACK_TYPE_DISTRIBUTION_BEFORE_SAMPLING.get(attack_type, 0)
            target_counts[attack_type] = min(requested, available)

    buckets = {attack_type: [] for attack_type in target_counts}
    collected_counts = {attack_type: 0 for attack_type in target_counts}

    for file_name in file_names:
        for chunk in iter_csv_member(file_name, chunksize=CHUNK_SIZE):
            chunk = clean_column_names(chunk)
            chunk = clean_invalid_values(chunk)
            chunk = chunk.copy()
            chunk['attackType'] = chunk['Label'].astype(str).map(map_label_to_attack_type)

            for attack_type, target in target_counts.items():
                remaining = target - collected_counts[attack_type]
                if remaining <= 0:
                    continue

                candidates = chunk[chunk['attackType'] == attack_type]
                if candidates.empty:
                    continue

                sample = candidates.sample(
                    n=min(remaining, len(candidates)), random_state=RANDOM_STATE
                )
                buckets[attack_type].append(sample)
                collected_counts[attack_type] += len(sample)

    sampled_frames = [pd.concat(parts) for parts in buckets.values() if parts]
    if not sampled_frames:
        return pd.DataFrame(), target_counts, collected_counts

    sample_df = pd.concat(sampled_frames, ignore_index=True)
    return sample_df, target_counts, collected_counts

sample_df, SAMPLE_TARGET_COUNTS, SAMPLE_COLLECTED_COUNTS = sample_rows_by_attack_type(SELECTED_FILES)
LABEL_DISTRIBUTION_AFTER_SAMPLING = sample_df['Label'].value_counts(dropna=False).to_dict()
ATTACK_TYPE_DISTRIBUTION_AFTER_SAMPLING = sample_df['attackType'].value_counts(dropna=False).to_dict()
BENIGN_SAMPLE_COUNT = int((sample_df['attackType'] == 'Benign').sum())
MALICIOUS_SAMPLE_COUNT = int((sample_df['attackType'] != 'Benign').sum())
pd.DataFrame({
    'target': SAMPLE_TARGET_COUNTS,
    'collected': SAMPLE_COLLECTED_COUNTS,
    'available': {key: ATTACK_TYPE_DISTRIBUTION_BEFORE_SAMPLING.get(key, 0) for key in SAMPLE_TARGET_COUNTS},
})

## 7. Convert Rows to Dashboard Alerts

In [ ]:
PROTOCOL_MAP = {
    0: 'HOPOPT',
    1: 'ICMP',
    6: 'TCP',
    17: 'UDP',
}

def map_label_to_attack_type(label: str) -> str:
    normalized = str(label).strip().lower()
    compact = ' '.join(normalized.replace('-', ' ').replace('_', ' ').split())
    squashed = compact.replace(' ', '')
    if normalized == 'benign':
        return 'Benign'
    if 'heartbleed' in compact:
        return 'Heartbleed'
    if 'ddos' in compact:
        return 'DDoS'
    if 'dos' in compact:
        return 'DoS'
    if 'bot' in compact:
        return 'Botnet'
    if 'web attack' in compact or 'sql injection' in compact or 'xss' in compact:
        return 'Web Attack'
    if 'brute' in compact and 'web' in compact:
        return 'Web Attack'
    if 'ftp' in compact and ('brute' in compact or 'bruteforce' in squashed):
        return 'Brute Force'
    if 'ssh' in compact and ('brute' in compact or 'bruteforce' in squashed or 'patator' in compact):
        return 'Brute Force'
    if 'infilteration' in compact or 'infiltration' in compact:
        return 'Infiltration'
    return 'Unknown Attack'

def map_attack_type_to_severity(attack_type: str) -> str:
    return {
        'Benign': 'Low',
        'Brute Force': 'Medium',
        'Heartbleed': 'Critical',
        'Web Attack': 'High',
        'Botnet': 'High',
        'DoS': 'High',
        'DDoS': 'High',
        'Infiltration': 'Critical',
        'Unknown Attack': 'Medium',
    }.get(attack_type, 'Medium')

def map_attack_type_to_risk_score(attack_type: str) -> int:
    return {
        'Benign': 25,
        'Brute Force': 68,
        'Heartbleed': 90,
        'Web Attack': 76,
        'Botnet': 80,
        'DoS': 84,
        'DDoS': 88,
        'Infiltration': 92,
        'Unknown Attack': 64,
    }.get(attack_type, 64)

def safe_value(row, field, default='unknown'):
    if field not in row.index:
        return default
    value = row[field]
    if pd.isna(value):
        return default
    return value

def safe_int(value, default=0):
    try:
        return int(float(value))
    except (TypeError, ValueError):
        return default

def protocol_name(value):
    try:
        return PROTOCOL_MAP.get(int(value), str(value))
    except (TypeError, ValueError):
        return str(value)

def row_to_alert(row, index):
    label = str(safe_value(row, 'Label', 'Benign'))
    attack_type = map_label_to_attack_type(label)
    is_benign = attack_type == 'Benign'
    risk_score = map_attack_type_to_risk_score(attack_type)
    source_ip = str(safe_value(row, 'Src IP', 'unknown-source'))
    destination_ip = str(safe_value(row, 'Dst IP', 'unknown-destination'))
    port = safe_int(safe_value(row, 'Dst Port', 0), default=0)
    protocol = protocol_name(safe_value(row, 'Protocol', 'TCP'))

    return {
        'id': f'AL-{index:04d}',
        'timestamp': str(safe_value(row, 'Timestamp', 'unknown-time')),
        'sourceIp': source_ip,
        'destinationIp': destination_ip,
        'protocol': protocol,
        'port': port,
        'attackType': attack_type,
        'severity': map_attack_type_to_severity(attack_type),
        'confidence': risk_score,
        'baseRiskScore': risk_score,
        'currentRiskScore': risk_score,
        'status': 'Unreviewed',
        'groundTruth': 'benign' if is_benign else 'malicious',
        'similarityKey': attack_type.lower().replace(' ', '-'),
        'triggerReason': 'Flow pattern maps from a labelled CSE-CIC-IDS2018 scenario.',
        'evidence': f"Dataset label: {label}; destination port: {port}; protocol: {protocol}.",
        'recommendedAction': 'Review related traffic and confirm whether escalation is needed.',
    }

alerts = [row_to_alert(row, i + 1) for i, (_, row) in enumerate(sample_df.iterrows())]
alerts[:2]

## 8. Validate Alert Schema

In [ ]:
REQUIRED_FIELDS = [
    'id', 'timestamp', 'sourceIp', 'destinationIp', 'protocol', 'port',
    'attackType', 'severity', 'confidence', 'baseRiskScore',
    'currentRiskScore', 'status', 'groundTruth', 'similarityKey',
    'triggerReason', 'evidence', 'recommendedAction'
]
VALID_SEVERITIES = {'Critical', 'High', 'Medium', 'Low'}
VALID_GROUND_TRUTH = {'benign', 'malicious'}

def validate_alert(alert):
    missing = [field for field in REQUIRED_FIELDS if field not in alert]
    assert not missing, f"{alert.get('id')} missing fields: {missing}"
    assert alert['severity'] in VALID_SEVERITIES, f"Invalid severity: {alert['severity']}"
    assert alert['groundTruth'] in VALID_GROUND_TRUTH, f"Invalid groundTruth: {alert['groundTruth']}"
    assert alert['status'] == 'Unreviewed', f"Invalid status: {alert['status']}"
    assert 0 <= alert['confidence'] <= 100, f"Invalid confidence: {alert['confidence']}"
    assert 0 <= alert['baseRiskScore'] <= 100, f"Invalid baseRiskScore: {alert['baseRiskScore']}"
    assert 0 <= alert['currentRiskScore'] <= 100, f"Invalid currentRiskScore: {alert['currentRiskScore']}"
    assert isinstance(alert['port'], (int, float)), f"Port must be numeric: {alert['port']}"
    assert str(alert['similarityKey']).strip(), 'similarityKey must not be empty'

def validate_alerts(alerts):
    seen_ids = set()
    for alert in alerts:
        validate_alert(alert)
        assert alert['id'] not in seen_ids, f"Duplicate alert ID: {alert['id']}"
        seen_ids.add(alert['id'])
    return True

validate_alerts(alerts)
print(f'Validated {len(alerts)} alert objects.')

## 9. Export sample-alerts.json

In [ ]:
with OUTPUT_PATH.open('w', encoding='utf-8') as file:
    json.dump(alerts, file, indent=2)

print(f'Exported Colab output to: {OUTPUT_PATH}')
print('Review the file before copying it into stage-1/data/processed/sample-alerts.json.')
OUTPUT_PATH

## Preprocessing Summary

In [ ]:
print('CSV file used:', CSV_FILES_USED)
print('Rows loaded:', ROWS_LOADED)
print('Rows after cleaning:', ROWS_AFTER_CLEANING)
print('Rows sampled:', len(sample_df))
print('Benign sample count:', BENIGN_SAMPLE_COUNT)
print('Malicious sample count:', MALICIOUS_SAMPLE_COUNT)
print('Missing / invalid values removed:', ROWS_REMOVED_DURING_CLEANING)
print('Label distribution before sampling:', LABEL_DISTRIBUTION_BEFORE_SAMPLING)
print('Label distribution after sampling:', LABEL_DISTRIBUTION_AFTER_SAMPLING)
print('Attack type distribution before sampling:', ATTACK_TYPE_DISTRIBUTION_BEFORE_SAMPLING)
print('Attack type distribution after sampling:', ATTACK_TYPE_DISTRIBUTION_AFTER_SAMPLING)
print('Sample target counts:', SAMPLE_TARGET_COUNTS)
print('Sample collected counts:', SAMPLE_COLLECTED_COUNTS)
print('Output file:', OUTPUT_PATH)

## 10. Notes and Findings

Record dataset files used, sample size, label distribution, cleaning decisions, dropped or renamed columns, and the final output path here.

Initial local archive findings:

- Primary Colab source is KaggleHub dataset `solarmainframe/ids-intrusion-csv`.
- Local `archive.zip` remains supported as a fallback for offline inspection.
- `archive.zip` contains 10 CSV files.
- Most CSV files have 80 columns and do not include `Src IP` or `Dst IP`.
- `02-20-2018.csv` has 84 columns and includes `Flow ID`, `Src IP`, `Src Port`, and `Dst IP`.
- All inspected files include `Label` as the final column.